In [1]:
import numpy as np
import pandas as pd
pdidx = pd.IndexSlice
import matplotlib
from matplotlib import pyplot as plt
from matplotlib import colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import seaborn as sns
from numpy import random as rand
from scipy import *
import time as T
import sys
import json
from datetime import datetime
import os
import copy
import collections
from itertools import combinations

from sklearn.metrics import roc_curve, roc_auc_score, log_loss, accuracy_score, precision_recall_curve,\
                            average_precision_score, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import xgboost as xgb
from xgboost import XGBRegressor
from xgboost import plot_importance
import optuna as opt

import cv2

import torch 
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, Dataset


from gtda.homology import CubicalPersistence
from gtda.diagrams import PersistenceImage
IMG_ROOT = '/home/travis/Projects/football_event_data_generation/camera_classifier/data_generation/frame_data'


In [2]:
# import and do some light processing to frame label data

frame_data_import = pd.read_csv('/home/travis/Projects/football_event_data_generation/camera_classifier/data_generation/frame_data/Labeled_Frame_Data_temp.csv').rename(columns={"Frame #": "Frame"})
frame_data_import = frame_data_import[["Match", "Frame", "Camera Angle"]]
frame_data_import['Frame'] = frame_data_import['Frame'].astype(int)
frame_data_import['Camera Angle'] = frame_data_import['Camera Angle'].astype(int)

#fix that one frame in the Como match
wrong_frame = (frame_data_import['Match'] == 'Como_vs_Torino') & (frame_data_import['Frame'] == 781)
frame_data_import.loc[wrong_frame, "Frame"] = 761
##

frame_data_import = frame_data_import.set_index(["Match", "Frame"]).sort_index()
master_model_data = (
    frame_data_import.groupby(level="Match", group_keys=False)
      .apply(lambda g: g.reindex(
          pd.MultiIndex.from_product(
              [[g.index.get_level_values("Match")[0]],
               range(g.index.get_level_values("Frame").min(),
                     1_000)],
              names=["Match", "Frame"]
          )
      ))
)
master_model_data["Camera Angle"] = master_model_data.groupby(level="Match")["Camera Angle"].ffill().astype(int)
# BECAUSE CHATGPT IS AN ASSHOLE AND IM A JOKER WHO BELIEVED IT I HAVE TO DROP SEVILLA
# SINCE THE FRAMES WERE DELETED WHAT A CUNT
master_model_data = master_model_data.drop(index='Sevilla_vs_Athletic', level='Match')


FILE_DICT = {
    'Arsenal_vs_Manchester_City' : 'Ars_ManCit',
    'Bayern_vs_Augsburg' : 'frames_for_travis/bayern_frames_3fps',
    'Como_vs_Torino' : 'frames_for_travis/como_frames_3fps',
    'Fiorentina_vs_Cagliari' : 'Fior_Cagl',
    'Hamburg_vs_St_Pauli' : 'Ham_StPaul',
    'Legane_vs_Real_Valladolid' : 'Leg_VallD',
    'OM_vs_Lens' : 'frames_for_travis/OM_frames_3fps',
    # 'Sevilla_vs_Athletic' : 'Sev_Bil',
    'Strasbourg_vs_Auxere' : 'Stras_Aux',
    'AC_Milan_vs_Pisa' : 'AC_Mil_vs_Pisa',
    'Brentford_vs_Nottingham' : 'Brent_vs_Nott',
    'Burnley_vs_Fulham' : 'Burn_vs_Ful',
    'Hoffenheim_vs_Leipzig' : 'Hoff_vs_Leip',
    'Levante_vs_Elche' : 'Lev_vs_Elch',
    'Monaco_vs_Nice' : 'Mon_vs_Nic'
}

# EXT_DICT = {
#     'Arsenal_vs_Manchester_City' : 'jpg',
#     'Bayern_vs_Augsburg' : 'jpg',
#     'Como_vs_Torino' : 'jpg',
#     'Fiorentina_vs_Cagliari' : 'jpg',
#     'Hamburg_vs_St_Pauli' : 'jpg',
#     'Legane_vs_Real_Valladolid' : 'jpg',
#     'OM_vs_Lens' : 'jpg',
#     'Sevilla_vs_Athletic' : 'jpg',
#     'Strasbourg_vs_Auxere' : 'jpg',    

# }

#they're all jpgs now, too lazy to change structure everywhere
EXT_DICT = collections.defaultdict(lambda: 'jpg')

In [3]:
def build_X_y_from_labels(
    master_model_data,
    FILE_DICT,
    EXT_DICT,
    IMG_ROOT,
    matches=None,
    target_w=160,
    target_h=90,
):
    if matches is None:
        matches = master_model_data.index.get_level_values("Match").unique()

    X_list = []
    y_list = []
    meta_rows = []
    missing = []

    for match in matches:
        subdf = master_model_data.loc[pd.IndexSlice[match, :], :]
        frames = subdf.index.get_level_values("Frame")

        for frame in frames:
            label = int(master_model_data.loc[(match, frame), "Camera Angle"])

            img_path = os.path.join(
                IMG_ROOT,
                FILE_DICT[match],
                f"frame_{int(frame):06d}.{EXT_DICT[match]}"
            )

            img = cv2.imread(img_path)
            if img is None:
                missing.append((match, int(frame), img_path))
                continue

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (target_w, target_h), interpolation=cv2.INTER_AREA)

            X_list.append(img)
            y_list.append(label)
            meta_rows.append((match, int(frame)))

    if len(X_list) == 0:
        raise ValueError("No images were loaded. Check FILE_DICT / EXT_DICT / IMG_ROOT.")

    X_images = np.stack(X_list, axis=0)              # (N, H, W, 3)
    y = np.array(y_list, dtype=np.int64)             # (N,)
    meta = pd.DataFrame(meta_rows, columns=["Match", "Frame"])
    missing_df = pd.DataFrame(missing, columns=["Match", "Frame", "Path"])

    return X_images, y, meta, missing_df

In [5]:
def make_scalar_maps(img_rgb, blur_ksize=5, preblur=False):
    if img_rgb.dtype != np.uint8:
        if img_rgb.max() <= 1.0:
            img_uint8 = (255 * img_rgb).astype(np.uint8)
        else:
            img_uint8 = img_rgb.astype(np.uint8)
    else:
        img_uint8 = img_rgb

    gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY).astype(np.float32)

    # if preblur:
    #     gray = cv2.GaussianBlur(gray, (blur_ksize, blur_ksize), 0)

    sobel_x = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3).astype(np.float32)
    sobel_y = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3).astype(np.float32)

    return {
        "sobel_x": sobel_x,
        "sobel_y": sobel_y,
    }


def build_one_scalar_batch(images_rgb, map_name, blur_ksize=5, preblur=True):
    out = []
    for img in images_rgb:
        maps = make_scalar_maps(img, blur_ksize=blur_ksize, preblur=preblur)
        out.append(maps[map_name])
    return np.stack(out, axis=0)


class PHImageFeatureExtractorChunked:
    def __init__(
        self,
        homology_dimensions=(0, 1),
        pi_sigma=0.1,
        pi_n_bins=20,
        n_jobs=None,
        chunk_size=250,
        blur_ksize=5,
        preblur=True,
        verbose=True,
        map_names=("sobel_x", "sobel_y"),
        pilot_size=None,
    ):
        self.homology_dimensions = homology_dimensions
        self.pi_sigma = pi_sigma
        self.pi_n_bins = pi_n_bins
        self.n_jobs = n_jobs
        self.chunk_size = chunk_size
        self.blur_ksize = blur_ksize
        self.preblur = preblur
        self.verbose = verbose
        self.map_names = list(map_names)
        self.pilot_size = pilot_size if pilot_size is not None else chunk_size

        self.cp_dict = {}
        self.pi_dict = {}

    def _iter_chunks(self, X):
        n = len(X)
        for start in range(0, n, self.chunk_size):
            end = min(start + self.chunk_size, n)
            yield start, end, X[start:end]

    def fit(self, images_rgb):
        n = len(images_rgb)

        for map_idx, name in enumerate(self.map_names, start=1):
            if self.verbose:
                print(f"\n[{map_idx}/{len(self.map_names)}] FIT map = {name}")

            cp = CubicalPersistence(
                homology_dimensions=self.homology_dimensions,
                n_jobs=self.n_jobs,
            )

            pilot_end = min(self.pilot_size, n)
            pilot_imgs = images_rgb[:pilot_end]

            if self.verbose:
                print(f"  building pilot scalar batch on {pilot_end} images")

            scalar_batch = build_one_scalar_batch(
                pilot_imgs,
                map_name=name,
                blur_ksize=self.blur_ksize,
                preblur=self.preblur,
            )

            if self.verbose:
                print("  computing pilot diagrams")

            pilot_diagrams = cp.fit_transform(scalar_batch)

            if self.verbose:
                print(f"  pilot diagrams shape: {pilot_diagrams.shape}")
                print("  fitting persistence image on pilot diagrams")

            pi = PersistenceImage(
                sigma=self.pi_sigma,
                n_bins=self.pi_n_bins,
                n_jobs=self.n_jobs,
            )
            pi.fit(pilot_diagrams)

            self.cp_dict[name] = cp
            self.pi_dict[name] = pi

            del scalar_batch
            del pilot_diagrams

        return self

    def transform(self, images_rgb):
        n = len(images_rgb)
        feature_blocks = []

        for map_idx, name in enumerate(self.map_names, start=1):
            if self.verbose:
                print(f"\n[{map_idx}/{len(self.map_names)}] TRANSFORM map = {name}")

            cp = self.cp_dict[name]
            pi = self.pi_dict[name]

            feat_chunks = []

            for start, end, chunk in self._iter_chunks(images_rgb):
                if self.verbose:
                    print(f"  features on chunk {start}:{end} / {n}")

                scalar_batch = build_one_scalar_batch(
                    chunk,
                    map_name=name,
                    blur_ksize=self.blur_ksize,
                    preblur=self.preblur,
                )

                diagrams_chunk = cp.transform(scalar_batch)
                feats_chunk = pi.transform(diagrams_chunk)
                feats_chunk = feats_chunk.reshape(feats_chunk.shape[0], -1)
                feat_chunks.append(feats_chunk)

                del scalar_batch
                del diagrams_chunk

            feats = np.concatenate(feat_chunks, axis=0)
            feature_blocks.append(feats)

            if self.verbose:
                print(f"  feature block shape: {feats.shape}")

            del feat_chunks

        X_ph = np.concatenate(feature_blocks, axis=1)

        if self.verbose:
            print(f"\nFinal PH feature shape: {X_ph.shape}")

        return X_ph

    def fit_transform(self, images_rgb):
        self.fit(images_rgb)
        return self.transform(images_rgb)

In [6]:
X_images, y_PH, meta, missing_df = build_X_y_from_labels(master_model_data,
    FILE_DICT,
    EXT_DICT,
    IMG_ROOT,
    matches=None,
    target_w=160,
    target_h=90,
)

# edge_extractor = PHImageFeatureExtractorChunked(
#     homology_dimensions=(1,2),
#     map_names=("edge_mag",),
#     chunk_size=1000,
#     n_jobs=-1,
#     verbose=True,
# )

# X_ph = edge_extractor.fit_transform(X_images)

# print(X_ph.shape)   # (N, n_features)

extractor = PHImageFeatureExtractorChunked(
    homology_dimensions=(1,2),
    map_names=("sobel_x", "sobel_y"),
    chunk_size=1000,
    pilot_size=1000,
    # blur_ksize=5,
    # preblur=True,
    n_jobs=-1,
    verbose=True,
)

X_ph = extractor.fit_transform(X_images)
print(X_ph.shape)


[1/2] FIT map = sobel_x
  building pilot scalar batch on 1000 images
  computing pilot diagrams
  pilot diagrams shape: (1000, 1044, 3)
  fitting persistence image on pilot diagrams

[2/2] FIT map = sobel_y
  building pilot scalar batch on 1000 images
  computing pilot diagrams
  pilot diagrams shape: (1000, 833, 3)
  fitting persistence image on pilot diagrams

[1/2] TRANSFORM map = sobel_x
  features on chunk 0:1000 / 14000
  features on chunk 1000:2000 / 14000
  features on chunk 2000:3000 / 14000
  features on chunk 3000:4000 / 14000
  features on chunk 4000:5000 / 14000
  features on chunk 5000:6000 / 14000
  features on chunk 6000:7000 / 14000
  features on chunk 7000:8000 / 14000
  features on chunk 8000:9000 / 14000
  features on chunk 9000:10000 / 14000
  features on chunk 10000:11000 / 14000
  features on chunk 11000:12000 / 14000
  features on chunk 12000:13000 / 14000
  features on chunk 13000:14000 / 14000
  feature block shape: (14000, 800)

[2/2] TRANSFORM map = sobel_y

In [7]:
from xgboost import XGBClassifier
from itertools import combinations

all_matches = list(FILE_DICT.keys())
# test_matches = ['Hamburg_vs_St_Pauli', 'OM_vs_Lens']
for test_matches in combinations(all_matches,2):
    print("RUNNING: ",test_matches)
    test_mask = meta['Match'].isin(test_matches).to_numpy()
    train_mask = ~test_mask
    
    X_train = X_ph[train_mask]
    X_test  = X_ph[test_mask]
    y_train = y_PH[train_mask]
    y_test  = y_PH[test_mask]
    
    clf = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=1000,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        random_state=42,
        early_stopping_rounds=30,
    )
    
    clf.fit(
        X_train,
        y_train,
        eval_set=[(X_test, y_test)],
        verbose=20,
    )
    
    y_pred = clf.predict(X_test)
    
    print("accuracy =", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    print(confusion_matrix(y_test, y_pred))
    print('===='*10)

RUNNING:  ('Arsenal_vs_Manchester_City', 'Bayern_vs_Augsburg')
[0]	validation_0-logloss:0.67852
[20]	validation_0-logloss:0.50428
[40]	validation_0-logloss:0.44397
[60]	validation_0-logloss:0.42142
[80]	validation_0-logloss:0.41994
[95]	validation_0-logloss:0.43436
accuracy = 0.856
              precision    recall  f1-score   support

           0       0.97      0.84      0.90      1552
           1       0.62      0.92      0.74       448

    accuracy                           0.86      2000
   macro avg       0.80      0.88      0.82      2000
weighted avg       0.89      0.86      0.86      2000

[[1300  252]
 [  36  412]]
RUNNING:  ('Arsenal_vs_Manchester_City', 'Como_vs_Torino')
[0]	validation_0-logloss:0.67772
[20]	validation_0-logloss:0.49434
[40]	validation_0-logloss:0.41082
[60]	validation_0-logloss:0.36821
[80]	validation_0-logloss:0.34256
[100]	validation_0-logloss:0.33047
[120]	validation_0-logloss:0.32230
[140]	validation_0-logloss:0.31835
[160]	validation_0-logloss:0.3

KeyboardInterrupt: 

In [ ]:
from xgbooy_prob = clf_S_M.predict_proba(X_test)[:, 1]st import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from itertools import combinations
import numpy as np
import cv2

# ------------------------------------------------------------
# Build flattened edge-map features from X_images
# ------------------------------------------------------------
def build_edge_features(X_images, canny_lo=50, canny_hi=150, blur_ksize=None):
    edge_list = []

    for i, img in enumerate(X_images):
        if img.dtype != np.uint8:
            if img.max() <= 1.0:
                img_uint8 = (255 * img).astype(np.uint8)
            else:
                img_uint8 = img.astype(np.uint8)
        else:
            img_uint8 = img

        gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)

        if blur_ksize is not None:
            gray = cv2.GaussianBlur(gray, blur_ksize, 0)

        edges = cv2.Canny(gray, canny_lo, canny_hi).astype(np.float32) / 255.0
        edge_list.append(edges.reshape(-1))

    return np.stack(edge_list, axis=0)


# build X once
X_edge = build_edge_features(
    X_images,
    canny_lo=50,
    canny_hi=150,
    blur_ksize=None,   # set to None if you do not want blur
)

print("X_edge shape:", X_edge.shape)

# ------------------------------------------------------------
# Match-holdout loop
# ------------------------------------------------------------
all_matches = list(FILE_DICT.keys())

for test_matches in combinations(all_matches, 2):
    print("RUNNING:", test_matches)

    test_mask = meta['Match'].isin(test_matches).to_numpy()
    train_mask = ~test_mask

    X_train = X_edge[train_mask]
    X_test  = X_edge[test_mask]
    y_train = y[train_mask]
    y_test  = y[test_mask]

    clf = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=1000,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        random_state=42,
        early_stopping_rounds=30,
    )

    clf.fit(
        X_train,
        y_train,
        eval_set=[(X_test, y_test)],
        verbose=20,
    )

    y_pred = clf.predict(X_test)

    print("accuracy =", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    print(confusion_matrix(y_test, y_pred))
    print("====" * 10)

X_edge shape: (9000, 14400)
RUNNING: ('Arsenal_vs_Manchester_City', 'Bayern_vs_Augsburg')
[0]	validation_0-logloss:0.68520
[20]	validation_0-logloss:0.57033
[40]	validation_0-logloss:0.50008
[60]	validation_0-logloss:0.45721
[80]	validation_0-logloss:0.42859
[100]	validation_0-logloss:0.40406
[120]	validation_0-logloss:0.38693
[140]	validation_0-logloss:0.37251
[160]	validation_0-logloss:0.36255
[180]	validation_0-logloss:0.35713
[200]	validation_0-logloss:0.35478
[220]	validation_0-logloss:0.35115
[240]	validation_0-logloss:0.34836
[260]	validation_0-logloss:0.34592
[280]	validation_0-logloss:0.34454
[300]	validation_0-logloss:0.34275
[320]	validation_0-logloss:0.34269
[331]	validation_0-logloss:0.34362
accuracy = 0.85
              precision    recall  f1-score   support

           0       0.94      0.86      0.90      1552
           1       0.63      0.81      0.71       448

    accuracy                           0.85      2000
   macro avg       0.78      0.84      0.80      200

In [ ]:
class BuildBoxDataset(Dataset):
    def __init__(
        self,
        master_model_data,
        file_dict
        boxes_root,
        matches,
    ):
        self.boxes_root = boxes_root
        self.file_dict = file_dict
        self.prefix = 'frames_for_travis/'

        rows = []
        for match in matches:
            subdf = master_model_data.loc[pd.IndexSlice[match, :], :]
            frames = subdf.index.get_level_values("Frame")

            for frame in frames:
                label = int(master_model_data.loc[(match, frame), "Camera Angle"])
                rows.append((match, int(frame), label))

        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def _load_boxes(self, match, frame):
        local_dir =  self.file_dict[match]
        if local_dir.startswith(self.prefix):
            local_dir = local_dir[len(prefix):]
        
        box_path = os.path.join(
            self.box_root.replace('###',local_dir),
            local_dir,
            f"frame_{frame:06d}.txt"
        )

        boxes = np.loadtxt(box_path,dtype=np.float32,usecols=(1,2,3,4))
        return boxes
        
    def __getitem__(self, idx):
        match, frame, label = self.rows[idx]

        boxes = self._load_boxes(match, frame) 
        areas = [w*h for w,h in zip(test[:,2],test[:,3])]

        num_of_detections = boxes.shape[0]
        avg_box_size = np.mean(areas)
        var_box_size = np.var(areas)
        max_box_size = np.max(areas)
        min_box_size = np.min(areas)
        avg_box_x = np.mean((test[:,0]))
        avg_box_y = np.mean((test[:,1]))

        x = torch.tensor([num_of_detections,
                          avg_box_size,
                          var_box_size,
                          max_box_size,
                          min_box_size,
                          avg_box_x,
                          avg_box_y],dtype=torch.float32)
        
        y = torch.tensor(label, dtype=torch.float32)

        return x, y


In [10]:
def make_camera_angle_loaders_boxes(
    master_model_data,
    file_dict,
    boxes_root,
    test_matches,
):
    all_matches = master_model_data.index.get_level_values("Match").unique().tolist()
    train_matches = [m for m in all_matches if m not in test_matches]

    trainset = CameraAngleDataset(
        master_model_data=master_model_data,
        file_dict=file_dict,
        boxes_root=boxes_root,
        matches=train_matches,
    )

    testset = CameraAngleDataset(
        master_model_data=master_model_data,
        file_dict=file_dict,
        boxes_root=boxes_root,
        matches=test_matches,
    )

    trainloader = DataLoader(
        trainset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=False,
    )

    testloader = DataLoader(
        testset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=False,
    )

    return trainloader, testloader

In [11]:
def match_box_stats(box_root,matches,master_model_data,file_dict):
    import warnings
    BOXES = []
    prefix = 'frames_for_travis/'
    for match in matches:
        frames = master_model_data.loc[pd.IndexSlice[match, :], :].index.get_level_values("Frame")
        local_dir =  file_dict[match]
        if local_dir.startswith(prefix):
            local_dir = local_dir[len(prefix):]
            
        for frame in frames:
            box_path = os.path.join(
                box_root.replace('###',local_dir),
                f"frame_{frame:06d}.txt"
            )
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                boxes = np.loadtxt(box_path,dtype=np.float32,usecols=(1,2,3,4))
            # print(box_path)
            # break
            boxes = np.atleast_2d(boxes)
            if boxes.size > 0:
                # print(boxes)
                areas = [w*h for w,h in zip(boxes[:,2],boxes[:,3])]
                # print(areas)
                num_of_detections = boxes.shape[0]
                # print(len(areas))
                avg_box_size = np.mean(areas)
                var_box_size = np.var(areas)
                max_box_size = np.max(areas)
                min_box_size = np.min(areas)
                avg_box_x = np.mean((boxes[:,0]))
                avg_box_y = np.mean((boxes[:,1]))
            else:
                num_of_detections = 0
                avg_box_size = 0
                var_box_size = float('inf')
                max_box_size = float('inf')
                min_box_size = 0
                avg_box_x = 0
                avg_box_y = 0
                
            cam = int(master_model_data.loc[(match, frame), "Camera Angle"])
            x = [frame, num_of_detections, avg_box_size, var_box_size, max_box_size, min_box_size, avg_box_x, avg_box_y, cam]
            BOXES.append([match] + x)

    return pd.DataFrame(BOXES,columns=['match','frame',
                                       'num_of_detections',
                                       'avg_box_size',
                                       'var_box_size', 'max_box_size',
                                       'min_box_size', 'avg_box_x',
                                       'avg_box_y', 'cam'])

In [12]:
boxes_dir = '/home/travis/Projects/football_event_data_generation/camera_classifier/data_generation/scripts/runs/boxes_only/###/labels'
all_matches = list(FILE_DICT.keys())
data = match_box_stats(boxes_dir,all_matches,master_model_data,FILE_DICT)

feature_cols = [
    c for c in data.columns
    if c not in ['match', 'frame', 'cam']
]

X = data[feature_cols].to_numpy()
y = data['cam'].to_numpy()

for test_matches in combinations(all_matches, 2):
    print("RUNNING:", test_matches)

    test_mask = data['match'].isin(test_matches).to_numpy()
    train_mask = ~test_mask

    X_train = X[train_mask]
    X_test  = X[test_mask]
    y_train = y[train_mask]
    y_test  = y[test_mask]

    clf = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=1000,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        # random_state=42,
        early_stopping_rounds=30,
    )

    clf.fit(
        X_train,
        y_train,
        eval_set=[(X_test, y_test)],
        verbose=20,
    )

    y_pred = clf.predict(X_test)

    print("accuracy =", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    print(confusion_matrix(y_test, y_pred))
    print("====" * 10)


KeyboardInterrupt: 

# Game comparison

In [13]:
boxes_dir = '/home/travis/Projects/football_event_data_generation/camera_classifier/data_generation/scripts/runs/boxes_only/###/labels'
all_matches = list(FILE_DICT.keys())
data = match_box_stats(boxes_dir, all_matches, master_model_data, FILE_DICT)

feature_cols = [c for c in data.columns if c not in ['match', 'frame', 'cam']]

X = data[feature_cols].to_numpy()
y = data['cam'].to_numpy()

test_matches = ['Arsenal_vs_Manchester_City', 'OM_vs_Lens']
test_mask = data['match'].isin(test_matches).to_numpy()
train_mask = ~test_mask

X_train = X[train_mask]
X_test  = X[test_mask]
y_train = y[train_mask]
y_test  = y[test_mask]


test_meta = data.loc[test_mask, ['match', 'frame']].copy()


clf_box = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    early_stopping_rounds=30,
)

clf_box.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=20,
)

y_pred = clf_box.predict(X_test)
y_prob = clf_box.predict_proba(X_test)[:, 1]

print("accuracy =", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print("====" * 10)

pred_df = test_meta.copy()
pred_df['truth'] = y_test
pred_df['pred'] = y_pred
pred_df['prob_1'] = y_prob
pred_df['correct'] = (pred_df['pred'] == pred_df['truth']).astype(int)

pred_df = pred_df.sort_values(['match', 'frame']).reset_index(drop=True)

save_path = './box_holdout_predictions.csv'
pred_df.to_csv(save_path, index=False)

print(f"Saved predictions to:\n{save_path}")
print(pred_df.head(20))

[0]	validation_0-logloss:0.66852
[20]	validation_0-logloss:0.37771
[40]	validation_0-logloss:0.26427
[60]	validation_0-logloss:0.21424
[80]	validation_0-logloss:0.19221
[100]	validation_0-logloss:0.18180
[120]	validation_0-logloss:0.17595
[140]	validation_0-logloss:0.17471
[160]	validation_0-logloss:0.17429
[163]	validation_0-logloss:0.17447
accuracy = 0.944
              precision    recall  f1-score   support

           0       0.93      1.00      0.96      1360
           1       1.00      0.83      0.90       640

    accuracy                           0.94      2000
   macro avg       0.96      0.91      0.93      2000
weighted avg       0.95      0.94      0.94      2000

[[1358    2]
 [ 110  530]]
Saved predictions to:
./box_holdout_predictions.csv
                         match  frame  truth  pred    prob_1  correct
0   Arsenal_vs_Manchester_City      0      0     1  0.728103        0
1   Arsenal_vs_Manchester_City      1      0     0  0.103759        1
2   Arsenal_vs_Manchest

In [15]:
test_mask = meta['Match'].isin(test_matches).to_numpy()
train_mask = ~test_mask

X_train = X_ph[train_mask]
X_test  = X_ph[test_mask]
y_train = y_PH[train_mask]
y_test  = y_PH[test_mask]

clf_ph = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    # random_state=42,
    early_stopping_rounds=30,
)

clf_ph.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=20,
)

y_pred = clf_ph.predict(X_test)
y_prob = clf_ph.predict_proba(X_test)[:, 1]

print("accuracy =", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print('===='*10)

pred_df = test_meta.copy()
pred_df['truth'] = y_test
pred_df['pred'] = y_pred
pred_df['prob_1'] = y_prob
pred_df['correct'] = (pred_df['pred'] == pred_df['truth']).astype(int)

pred_df = pred_df.sort_values(['match', 'frame']).reset_index(drop=True)

save_path = './PH_holdout_predictions.csv'
pred_df.to_csv(save_path, index=False)

print(f"Saved predictions to:\n{save_path}")
print(pred_df.head(20))

[0]	validation_0-logloss:0.67600
[20]	validation_0-logloss:0.47324
[40]	validation_0-logloss:0.38923
[60]	validation_0-logloss:0.35170
[80]	validation_0-logloss:0.33108
[100]	validation_0-logloss:0.32339
[120]	validation_0-logloss:0.32412
[140]	validation_0-logloss:0.32690
[142]	validation_0-logloss:0.32755
accuracy = 0.894
              precision    recall  f1-score   support

           0       0.89      0.96      0.92      1360
           1       0.90      0.75      0.82       640

    accuracy                           0.89      2000
   macro avg       0.90      0.86      0.87      2000
weighted avg       0.89      0.89      0.89      2000

[[1306   54]
 [ 158  482]]
Saved predictions to:
./PH_holdout_predictions.csv
                         match  frame  truth  pred    prob_1  correct
0   Arsenal_vs_Manchester_City      0      0     0  0.111698        1
1   Arsenal_vs_Manchester_City      1      0     0  0.104249        1
2   Arsenal_vs_Manchester_City      2      0     0  0.24301